In [1]:
# arregar dados

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display, Markdown

EVENTS_PATH = Path(
    "../data/processed/microservices_sample/"
    "microservices_traces_anonymized.csv"
)

FEATURES_PATH = Path(
    "../data/processed/microservices_sample/"
    "microservices_trace_features.csv"
)

LABELS_PATH = Path(
    "../data/processed/microservices_sample/"
    "microservices_trace_manual_labels.csv"
)

events = pd.read_csv(
    EVENTS_PATH,
    dtype=str,
    keep_default_na=False,
)

features = pd.read_csv(FEATURES_PATH)

events["level"] = events["level"].str.upper()

In [2]:
# reconstruir os perfis

def signal_profile(row):
    profile = []

    if row["error_count"] > 0:
        profile.append("ERROR")

    if row["warn_count"] > 0:
        profile.append("WARN")

    if row["has_stack_trace"] > 0:
        profile.append("STACK_TRACE")

    if row["text_signal_count"] > 0:
        profile.append("TEXT")

    return "+".join(profile) if profile else "NONE"


features["signal_profile"] = features.apply(
    signal_profile,
    axis=1,
)

profile_counts = (
    features["signal_profile"]
    .value_counts()
    .rename_axis("signal_profile")
    .reset_index(name="trace_count")
)

profile_counts.head(30)

,signal_profile,trace_count
0,NONE,14596
1,WARN,2064
2,ERROR+WARN,2032
3,ERROR,955
4,ERROR+STACK_TRACE+TEXT,146
5,WARN+TEXT,69
6,ERROR+WARN+STACK_TRACE+TEXT,34
7,ERROR+WARN+TEXT,31
8,TEXT,27
9,ERROR+WARN+STACK_TRACE,20


In [3]:
# criar fila de revisão

MAX_PROFILES = 20
TRACES_PER_PROFILE = 8
SEED = 20260914

profiles_to_review = (
    profile_counts
    .head(MAX_PROFILES)["signal_profile"]
    .tolist()
)

if "NONE" not in profiles_to_review:
    profiles_to_review.append("NONE")

review_queue = []

for profile in profiles_to_review:
    candidates = features[
        features["signal_profile"] == profile
    ]

    sample = candidates.sample(
        n=min(TRACES_PER_PROFILE, len(candidates)),
        random_state=SEED,
    )

    review_queue.append(sample)

review_queue = pd.concat(
    review_queue,
    ignore_index=True,
)

review_queue = review_queue[
    ["tc_trace_id", "signal_profile", "event_count",
    "error_count", "warn_count", "service_count"]
]

review_queue

,tc_trace_id,signal_profile,event_count,error_count,warn_count,service_count
0,e662ef37-5aaa-4c55-9376-60986e9f2d43,NONE,4,0,0,1
1,58fa3853-67d4-4f01-8ff6-dec79d91dc18,NONE,3,0,0,1
2,160f78d0-80cf-4a42-8b98-e6f3edeca689,NONE,1,0,0,1
3,c2e1254b-f7a2-40af-86f2-64e6672db87e,NONE,7,0,0,2
4,e74cf56d-7942-462a-a4a8-7a03c7ea1d3c,NONE,1,0,0,1
...,...,...,...,...,...,...
77,5ea9e3c8-5717-40ea-9590-683a49ea4893,ERROR+WARN+STACK_TRACE,48,1,6,2
78,70492f90-cbff-4bb0-8a12-901a667bce09,ERROR+WARN+STACK_TRACE,49,1,6,2
79,513406ec-0f0b-4dcb-8251-e64c3026414e,ERROR+WARN+STACK_TRACE,107,1,5,11
80,8073ae0d-7fc3-4639-ada1-f300d2a4da81,ERROR+STACK_TRACE,1,1,0,1


In [12]:
# consultar um trace completo

def show_trace(trace_id: str) -> None:
    trace = events[
        events["tc_trace_id"] == trace_id
    ].copy()

    if trace.empty:
        print("Trace não encontrado.")
        return

    profile = features.loc[
        features["tc_trace_id"] == trace_id,
        "signal_profile",
    ].iloc[0]

    display(Markdown(
        f"### Trace: `{trace_id}`\n"
        f"**Perfil de sinais:** `{profile}`\n"
        f"**Eventos:** {len(trace)}"
    ))

    columns = [
        "@timestamp",
        "level",
        "tc_service",
        "tc_app_name",
        "logger_name",
        "message",
        "stack_trace",
    ]

    with pd.option_context(
        "display.max_colwidth",
        None,
        "display.max_rows",
        None,
        "display.width",
        0,
    ):
        display(
            trace[columns].sort_values("@timestamp")
        )


### Trace: `26d60157-93ce-45a3-a577-d07eb7ba64c0`
**Perfil de sinais:** `WARN`
**Eventos:** 1

,@timestamp,level,tc_service,tc_app_name,logger_name,message,stack_trace
123817,2026-09-11T15:43:49.908Z,WARN,security,platform-security-backend,br.com.triersistemas.cloud.platform.security.token.CheckTokenImpl,"rememberMeValue has no token that match with given token: '<JWT_164>', remember-me tokens: 'false'.",


In [ ]:
### registrar um rótulo manual

def save_manual_label(
    trace_id: str,
    label: str,
    notes: str = "",
) -> None:
    valid_labels = {
        "anomalia",
        "nao_anomalia",
        "incerto",
    }

    if label not in valid_labels:
        raise ValueError(
            f"Use uma destas opções: {valid_labels}"
        )

    profile = features.loc[
        features["tc_trace_id"] == trace_id,
        "signal_profile",
    ].iloc[0]

    row = pd.DataFrame(
        [
            {
                "tc_trace_id": trace_id,
                "signal_profile": profile,
                "label": label,
                "notes": notes,
                "reviewed_at": datetime.now(
                    timezone.utc
                ).isoformat(),
            }
        ]
    )

    if LABELS_PATH.exists():
        previous = pd.read_csv(
            LABELS_PATH,
            dtype=str,
        )

        previous = previous[
            previous["tc_trace_id"] != trace_id
        ]

        labels = pd.concat(
            [previous, row],
            ignore_index=True,
        )
    else:
        labels = row

    labels.to_csv(
        LABELS_PATH,
        index=False,
    )

    print(f"Rótulo salvo: {trace_id}")

# Exemplo:



Rótulo salvo: 58fa3853-67d4-4f01-8ff6-dec79d91dc18


In [133]:
#Para consultar um trace:

TRACE_ID = review_queue.iloc[59]["tc_trace_id"]

show_trace(TRACE_ID)

### Trace: `7e279ac4-23b7-4fcf-9580-d8325becc3ee`
**Perfil de sinais:** `ERROR+WARN+TEXT`
**Eventos:** 307

,@timestamp,level,tc_service,tc_app_name,logger_name,message,stack_trace
91625,2026-09-11T15:06:22.808480824Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.EntityInitialLoadHandlerImpl,"A carga inicial para o tenant 'franfarma15927' foi iniciada para as entidades: cidadeEmpresa, cidadeEmpresa, empresa, empresa, estadoEmpresa, estadoEmpresa, tenant, tenant...",
137383,2026-09-11T15:06:22.808668034Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.EntityInitialLoadHandlerImpl,"Iniciando a carga inicial em paralelo, do tenant: 'franfarma15927' para as entidades: estadoEmpresa, tenant...",
137384,2026-09-11T15:06:22.809688947Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.entity.estadoempresa.EstadoEmpresaInitialLoadFdwImpl,Force: false. Tenant 'franfarma15927' - Carga inicial da entidade 'estadoEmpresa'. Iniciando...,
143006,2026-09-11T15:06:22.809909765Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.entity.tenant.TenantInitialLoadFdwImpl,Force: false. Tenant 'franfarma15927' - Carga inicial da entidade 'tenant'. Iniciando...,
91626,2026-09-11T15:06:22.811873038Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.entity.tenant.TenantInitialLoadFdwImpl,Force: false. Tenant 'franfarma15927' - Carga inicial da entidade 'tenant'. ABORTADO! Processo já finalizado em 2023-05-31T20:30:26.942271,
116516,2026-09-11T15:06:22.811876806Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.entity.estadoempresa.EstadoEmpresaInitialLoadFdwImpl,Force: false. Tenant 'franfarma15927' - Carga inicial da entidade 'estadoEmpresa'. ABORTADO! Processo já finalizado em 2023-04-26T19:52:35.342290,
137385,2026-09-11T15:06:22.812335859Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.EntityInitialLoadHandlerImpl,"Carga inicial finalizada para tenant 'franfarma15927', EntityInitialLoadResult(entityName=estadoEmpresa, recordsLoaded=0)",
97545,2026-09-11T15:06:22.812374236Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.EntityInitialLoadHandlerImpl,"Carga inicial finalizada para tenant 'franfarma15927', EntityInitialLoadResult(entityName=tenant, recordsLoaded=0)",
97546,2026-09-11T15:06:22.812388764Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.EntityInitialLoadHandlerImpl,A carga inicial em paralelo para o tenant 'franfarma15927' foi executada em 4 ms,
91627,2026-09-11T15:06:22.812444391Z,INFO,cadastrofacilitado,integration-cadastrofacilitado-backend,br.com.triersistemas.cloud.integration.cadastrofacilitado.EntityInitialLoadHandlerImpl,"Iniciando a carga inicial serial, do tenant: 'franfarma15927', das entidades: cidadeEmpresa, empresa...",


In [134]:
#  "anomalia",
#         "nao_anomalia",
#         "incerto",
# Bussiness Validation"

save_manual_label(
    TRACE_ID,
    label="anomalia",
    notes="task em execução/órfã",
)

Rótulo salvo: 7e279ac4-23b7-4fcf-9580-d8325becc3ee
